# GDA

## Multivariate Gaussian distribution

In [ ]:
"""
Visualize two 2D Gaussian distributions, given their means and
covariance matrices as inputs.

Inputs:  MU1, SIGMA1, MU2, SIGMA2
"""

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers the 3D projection)

N_SAMPLES = 300
MU1 = np.array([-1.0, -1.0])
SIGMA1 = np.array([[1.5, 0.6],
                     [0.6, 1.0]])

MU2 = np.array([2.0, 2.0])
SIGMA2 = np.array([[2.0, -1.0],
                     [-1.0, 1.5]])

SEED = 0


def gaussian_pdf(X_grid, mu, sigma):
    d = mu.shape[0]
    sigma_inv = np.linalg.inv(sigma)
    sigma_det = np.linalg.det(sigma)
    norm_const = 1.0 / np.sqrt((2 * np.pi) ** d * sigma_det)
    diff = X_grid - mu
    exponent = -0.5 * np.einsum("...i,ij,...j->...", diff, sigma_inv, diff)
    return norm_const * np.exp(exponent)


def plot_eigenvectors(ax, mu, sigma, color, scale=2.0):
    """Draw the eigenvectors of sigma as arrows from mu, scaled by the
    square root of their eigenvalues (i.e. by the standard deviation
    along that axis), so the arrow lengths reflect the actual spread
    of the distribution along each principal axis."""
    eigenvalues, eigenvectors = np.linalg.eigh(sigma)
    for eigval, eigvec in zip(eigenvalues, eigenvectors.T):
        length = scale * np.sqrt(eigval)
        ax.annotate(
            "", xy=mu + length * eigvec, xytext=mu,
            arrowprops=dict(arrowstyle="->", color=color, linewidth=2),
        )


def main():
    rng = np.random.default_rng(SEED)
    X1 = rng.multivariate_normal(MU1, SIGMA1, N_SAMPLES)
    X2 = rng.multivariate_normal(MU2, SIGMA2, N_SAMPLES)

    for name, sigma in [("Sigma1", SIGMA1), ("Sigma2", SIGMA2)]:
        sigma_inv = np.linalg.inv(sigma)
        eigenvalues, eigenvectors = np.linalg.eigh(sigma)
        print(f"{name} =\n{sigma}")
        print(f"{name}^-1 =\n{sigma_inv}")
        print(f"eigenvalues of {name}: {eigenvalues}")
        print(f"eigenvectors of {name} (columns):\n{eigenvectors}\n")

    x_min = min(X1[:, 0].min(), X2[:, 0].min()) - 1
    x_max = max(X1[:, 0].max(), X2[:, 0].max()) + 1
    y_min = min(X1[:, 1].min(), X2[:, 1].min()) - 1
    y_max = max(X1[:, 1].max(), X2[:, 1].max()) + 1
    x_grid, y_grid = np.meshgrid(
        np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300)
    )
    grid_points = np.stack([x_grid, y_grid], axis=-1)

    pdf1 = gaussian_pdf(grid_points, MU1, SIGMA1)
    pdf2 = gaussian_pdf(grid_points, MU2, SIGMA2)

    fig = plt.figure(figsize=(19, 6))
    ax_scatter = fig.add_subplot(1, 3, 1)
    ax_contour = fig.add_subplot(1, 3, 2)
    ax_3d = fig.add_subplot(1, 3, 3, projection="3d")
    # 2D scatter plot of samples from both distributions.
    ax_scatter.scatter(X1[:, 0], X1[:, 1], alpha=0.5, color="tab:blue", label="Dist. 1")
    ax_scatter.scatter(X2[:, 0], X2[:, 1], alpha=0.5, color="tab:orange", label="Dist. 2")
    ax_scatter.scatter(*MU1, color="navy", marker="x", s=100, label="$\\mu_1$")
    ax_scatter.scatter(*MU2, color="darkred", marker="x", s=100, label="$\\mu_2$")
    plot_eigenvectors(ax_scatter, MU1, SIGMA1, "navy")
    plot_eigenvectors(ax_scatter, MU2, SIGMA2, "darkred")
    ax_scatter.axhline(0, color="grey", linewidth=0.8, linestyle=":")
    ax_scatter.axvline(0, color="grey", linewidth=0.8, linestyle=":")
    ax_scatter.set_xlim(x_min, x_max)
    ax_scatter.set_ylim(y_min, y_max)
    ax_scatter.set_xlabel("$x_1$")
    ax_scatter.set_ylabel("$x_2$")
    ax_scatter.set_title("Samples from both Gaussian distributions")
    ax_scatter.legend()

    # Contour plot of both densities.
    ax_contour.contour(x_grid, y_grid, pdf1, colors="tab:blue", linewidths=1.2, levels=6)
    ax_contour.contour(x_grid, y_grid, pdf2, colors="tab:orange", linewidths=1.2, levels=6)
    ax_contour.scatter(*MU1, color="navy", marker="x", s=100, label="$\\mu_1$")
    ax_contour.scatter(*MU2, color="darkred", marker="x", s=100, label="$\\mu_2$")
    plot_eigenvectors(ax_contour, MU1, SIGMA1, "navy")
    plot_eigenvectors(ax_contour, MU2, SIGMA2, "darkred")
    ax_contour.axhline(0, color="grey", linewidth=0.8, linestyle=":")
    ax_contour.axvline(0, color="grey", linewidth=0.8, linestyle=":")
    ax_contour.set_xlim(x_min, x_max)
    ax_contour.set_ylim(y_min, y_max)
    ax_contour.set_xlabel("$x_1$")
    ax_contour.set_ylabel("$x_2$")
    ax_contour.set_title("Density contours: $N(\\mu_1,\\Sigma_1)$ vs. $N(\\mu_2,\\Sigma_2)$")
    ax_contour.legend()

    # 3D surface plot: density (z-axis) over the (x1, x2) plane. Two
    # separate surfaces are drawn (rather than a single combined one),
    # since summing two densities directly is not itself a meaningful
    # density to visualize - the point here is comparing each
    # distribution's actual shape and height side by side.
    ax_3d.plot_surface(x_grid, y_grid, pdf1, cmap="Blues", alpha=0.75,
                        linewidth=0, antialiased=True)
    ax_3d.plot_surface(x_grid, y_grid, pdf2, cmap="Oranges", alpha=0.75,
                        linewidth=0, antialiased=True)
    ax_3d.set_xlabel("$x_1$")
    ax_3d.set_ylabel("$x_2$")
    ax_3d.set_zlabel("density")
    ax_3d.set_title("3D surface: $N(\\mu_1,\\Sigma_1)$ (blue) vs.\n$N(\\mu_2,\\Sigma_2)$ (orange)")

    fig.tight_layout()
    fig.savefig("two_gaussians.png", dpi=150)
    print("Saved plot to two_gaussians.png")


if __name__ == "__main__":
    main()

## GDA

In [ ]:
"""
Demo of Gaussian Discriminant Analysis (GDA) on two-dimensional data
with two classes y in {0, 1}.

Model (standard GDA, shared covariance across classes):
    y ~ Bernoulli(phi)
    x | y=0 ~ N(mu0, Sigma)
    x | y=1 ~ N(mu1, Sigma)

The class means and the (shared) covariance are estimated from data
by maximum likelihood, in closed form:
    phi  = (1/n) * sum(y_i == 1)
    mu0  = mean of x_i where y_i == 0
    mu1  = mean of x_i where y_i == 1
    Sigma = (1/n) * sum_i (x_i - mu_{y_i})(x_i - mu_{y_i})^T
            (pooled across both classes, using each point's own class mean)

Because both classes share the same covariance matrix, the GDA
decision boundary p(y=1|x) = p(y=0|x) is exactly linear, obtained from
the log-odds:
    log[p(y=1|x)/p(y=0|x)] = w^T x + b = 0
    w = Sigma^{-1} (mu1 - mu0)
    b = -0.5*mu1^T Sigma^{-1} mu1 + 0.5*mu0^T Sigma^{-1} mu0 + log(phi/(1-phi))

This is the same functional form (a linear boundary) that logistic
regression fits directly, which is why the two are compared here:
logistic regression is fit on the same data using scikit-learn, and
its boundary is drawn alongside GDA's for comparison.
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

N_PER_CLASS = 150
MU0 = np.array([-1.0, -1.0])
MU1 = np.array([2.0, 2.0])
SIGMA = np.array([[2.0, 0.8],
                   [0.8, 1.5]])   # shared covariance for both classes
SEED = 0


def generate_data(n_per_class=N_PER_CLASS, mu0=MU0, mu1=MU1, sigma=SIGMA, seed=SEED):
    rng = np.random.default_rng(seed)
    x0 = rng.multivariate_normal(mu0, sigma, n_per_class)
    x1 = rng.multivariate_normal(mu1, sigma, n_per_class)
    X = np.vstack([x0, x1])
    y = np.concatenate([np.zeros(n_per_class), np.ones(n_per_class)])
    return X, y


def fit_gda(X, y):
    n = len(y)
    phi = np.mean(y == 1)
    mu0 = X[y == 0].mean(axis=0)
    mu1 = X[y == 1].mean(axis=0)

    # Shared covariance, pooled across both classes.
    centered = np.where((y == 1)[:, None], X - mu1, X - mu0)
    Sigma = (centered.T @ centered) / n

    return phi, mu0, mu1, Sigma


def gaussian_pdf(X_grid, mu, sigma):
    """Explicit multivariate Gaussian density, evaluated over a grid of
    points. X_grid has shape (..., 2); mu has shape (2,); sigma has
    shape (2, 2)."""
    d = mu.shape[0]
    sigma_inv = np.linalg.inv(sigma)
    sigma_det = np.linalg.det(sigma)
    norm_const = 1.0 / (np.sqrt((2 * np.pi) ** d * sigma_det))

    diff = X_grid - mu
    exponent = -0.5 * np.einsum("...i,ij,...j->...", diff, sigma_inv, diff)
    return norm_const * np.exp(exponent)


def gda_decision_boundary_params(phi, mu0, mu1, sigma):
    sigma_inv = np.linalg.inv(sigma)
    w = sigma_inv @ (mu1 - mu0)
    b = (-0.5 * mu1 @ sigma_inv @ mu1
         + 0.5 * mu0 @ sigma_inv @ mu0
         + np.log(phi / (1 - phi)))
    return w, b


def boundary_line(w, b, x1_range):
    """For a linear boundary w1*x1 + w2*x2 + b = 0, solve for x2 as a
    function of x1."""
    return -(w[0] * x1_range + b) / w[1]


def main():
    X, y = generate_data()

    phi, mu0, mu1, Sigma = fit_gda(X, y)
    print(f"GDA estimates:")
    print(f"  phi = {phi:.4f}")
    print(f"  mu0 = {mu0}")
    print(f"  mu1 = {mu1}")
    print(f"  Sigma =\n{Sigma}")

    w_gda, b_gda = gda_decision_boundary_params(phi, mu0, mu1, Sigma)
    print(f"GDA decision boundary: w={w_gda}, b={b_gda:.4f}")

    # Logistic regression, fit on the same data, for comparison.
    # C=np.inf disables the L2 penalty (an effectively unregularized fit,
    # comparable to GDA's unregularized MLE); this sklearn version emits
    # a harmless internal warning here regardless of how "no
    # regularization" is requested, so it is suppressed.
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        logreg = LogisticRegression(C=np.inf)
        logreg.fit(X, y)
    w_log = logreg.coef_[0]
    b_log = logreg.intercept_[0]
    print(f"Logistic regression boundary: w={w_log}, b={b_log:.4f}")

    # Set up the plotting grid.
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    x1_grid, x2_grid = np.meshgrid(
        np.linspace(x1_min, x1_max, 200), np.linspace(x2_min, x2_max, 200)
    )
    grid_points = np.stack([x1_grid, x2_grid], axis=-1)

    pdf0 = gaussian_pdf(grid_points, mu0, Sigma)
    pdf1 = gaussian_pdf(grid_points, mu1, Sigma)

    x1_line = np.linspace(x1_min, x1_max, 100)

    plt.figure(figsize=(8, 6.5))
    plt.scatter(X[y == 0, 0], X[y == 0, 1], alpha=0.5, color="tab:blue", label="y=0")
    plt.scatter(X[y == 1, 0], X[y == 1, 1], alpha=0.5, color="tab:orange", label="y=1")

    plt.contour(x1_grid, x2_grid, pdf0, colors="tab:blue", linewidths=1, levels=6)
    plt.contour(x1_grid, x2_grid, pdf1, colors="tab:orange", linewidths=1, levels=6)

    plt.plot(x1_line, boundary_line(w_gda, b_gda, x1_line), color="black",
              linewidth=2, label="GDA decision boundary")
    plt.plot(x1_line, boundary_line(w_log, b_log, x1_line), color="green",
              linewidth=2, linestyle="--", label="logistic regression boundary")

    plt.xlim(x1_min, x1_max)
    plt.ylim(x2_min, x2_max)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.title("Gaussian Discriminant Analysis: class-conditional densities and\n"
              "decision boundary, compared with logistic regression")
    plt.legend()
    plt.tight_layout()
    plt.savefig("gda_demo.png", dpi=150)
    print("Saved plot to gda_demo.png")


if __name__ == "__main__":
    main()

## Quadratic fit

In [ ]:
"""
Demo of Gaussian Discriminant Analysis (GDA) and Quadratic Discriminant
Analysis (QDA) on two-dimensional data with two classes y in {0, 1},
where the two classes now have DIFFERENT covariance matrices - i.e.
standard GDA's shared-covariance assumption is violated by
construction. This demonstrates what happens when that assumption is
wrong, and how QDA (which drops the shared-covariance assumption)
recovers a better-fitting, curved decision boundary instead.

Models compared:

1. GDA (shared covariance, the "standard" GDA from before):
       x | y=0 ~ N(mu0, Sigma_pooled),   x | y=1 ~ N(mu1, Sigma_pooled)
   Sigma_pooled is estimated as a single covariance matrix pooled
   across both classes. Because both classes are forced to share one
   covariance matrix, the resulting decision boundary is exactly
   linear - even though the true data-generating process does not
   actually share a covariance matrix between classes.

2. QDA (separate covariance per class):
       x | y=0 ~ N(mu0, Sigma0),   x | y=1 ~ N(mu1, Sigma1)
   Sigma0 and Sigma1 are estimated separately from each class's own
   data. The decision boundary p(y=1|x) = p(y=0|x) is then quadratic
   in x (the Sigma0/Sigma1 terms no longer cancel the way they do
   when Sigma0 = Sigma1), so it is found numerically here as the zero
   level set of the log-odds discriminant function, rather than in a
   simple closed line-equation form.

3. Logistic regression, fit directly and discriminatively on the same
   data (scikit-learn), for comparison. Its decision boundary is
   linear by construction, regardless of the true covariance
   structure - unlike GDA/QDA, it makes no distributional assumption
   about x | y at all.
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

N_PER_CLASS = 150
MU0 = np.array([-1.0, -1.0])
MU1 = np.array([2.0, 2.0])
SIGMA0 = np.array([[1.0, 0.6],
                     [0.6, 1.0]])    # class 0: fairly round, positively correlated
SIGMA1 = np.array([[3.0, -1.2],
                     [-1.2, 0.8]])    # class 1: elongated, negatively correlated
SEED = 0


def generate_data(n_per_class=N_PER_CLASS, mu0=MU0, mu1=MU1,
                   sigma0=SIGMA0, sigma1=SIGMA1, seed=SEED):
    rng = np.random.default_rng(seed)
    x0 = rng.multivariate_normal(mu0, sigma0, n_per_class)
    x1 = rng.multivariate_normal(mu1, sigma1, n_per_class)
    X = np.vstack([x0, x1])
    y = np.concatenate([np.zeros(n_per_class), np.ones(n_per_class)])
    return X, y


def fit_gda_shared_covariance(X, y):
    """Standard GDA: one covariance matrix pooled across both classes."""
    n = len(y)
    phi = np.mean(y == 1)
    mu0 = X[y == 0].mean(axis=0)
    mu1 = X[y == 1].mean(axis=0)

    centered = np.where((y == 1)[:, None], X - mu1, X - mu0)
    sigma_pooled = (centered.T @ centered) / n

    return phi, mu0, mu1, sigma_pooled


def fit_qda(X, y):
    """QDA: separate covariance matrix estimated for each class."""
    phi = np.mean(y == 1)
    X0, X1 = X[y == 0], X[y == 1]
    mu0, mu1 = X0.mean(axis=0), X1.mean(axis=0)

    centered0 = X0 - mu0
    centered1 = X1 - mu1
    sigma0 = (centered0.T @ centered0) / len(X0)
    sigma1 = (centered1.T @ centered1) / len(X1)

    return phi, mu0, mu1, sigma0, sigma1


def gaussian_pdf(X_grid, mu, sigma):
    """Explicit multivariate Gaussian density over a grid of points."""
    d = mu.shape[0]
    sigma_inv = np.linalg.inv(sigma)
    sigma_det = np.linalg.det(sigma)
    norm_const = 1.0 / (np.sqrt((2 * np.pi) ** d * sigma_det))
    diff = X_grid - mu
    exponent = -0.5 * np.einsum("...i,ij,...j->...", diff, sigma_inv, diff)
    return norm_const * np.exp(exponent)


def gda_linear_boundary_params(phi, mu0, mu1, sigma):
    """Closed-form linear boundary for shared-covariance GDA."""
    sigma_inv = np.linalg.inv(sigma)
    w = sigma_inv @ (mu1 - mu0)
    b = (-0.5 * mu1 @ sigma_inv @ mu1
         + 0.5 * mu0 @ sigma_inv @ mu0
         + np.log(phi / (1 - phi)))
    return w, b


def boundary_line(w, b, x1_range):
    return -(w[0] * x1_range + b) / w[1]


def qda_discriminant(X_grid, phi, mu0, mu1, sigma0, sigma1):
    """log p(y=1|x) - log p(y=0|x), up to the additive constant that
    cancels between the two classes. Its zero level set is the QDA
    decision boundary; this is quadratic in x whenever sigma0 != sigma1,
    so (unlike the shared-covariance case) it is not a simple line."""
    log_pdf1 = np.log(gaussian_pdf(X_grid, mu1, sigma1)) + np.log(phi)
    log_pdf0 = np.log(gaussian_pdf(X_grid, mu0, sigma0)) + np.log(1 - phi)
    return log_pdf1 - log_pdf0


def main():
    X, y = generate_data()

    phi_g, mu0_g, mu1_g, sigma_pooled = fit_gda_shared_covariance(X, y)
    w_gda, b_gda = gda_linear_boundary_params(phi_g, mu0_g, mu1_g, sigma_pooled)
    print("GDA (shared covariance):")
    print(f"  Sigma_pooled =\n{sigma_pooled}")
    print(f"  boundary: w={w_gda}, b={b_gda:.4f}\n")

    phi_q, mu0_q, mu1_q, sigma0_hat, sigma1_hat = fit_qda(X, y)
    print("QDA (separate covariances):")
    print(f"  Sigma0_hat =\n{sigma0_hat}")
    print(f"  Sigma1_hat =\n{sigma1_hat}\n")

    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        logreg = LogisticRegression(C=np.inf)
        logreg.fit(X, y)
    w_log, b_log = logreg.coef_[0], logreg.intercept_[0]
    print(f"Logistic regression boundary: w={w_log}, b={b_log:.4f}")

    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    x1_grid, x2_grid = np.meshgrid(
        np.linspace(x1_min, x1_max, 300), np.linspace(x2_min, x2_max, 300)
    )
    grid_points = np.stack([x1_grid, x2_grid], axis=-1)

    pdf0 = gaussian_pdf(grid_points, mu0_q, sigma0_hat)
    pdf1 = gaussian_pdf(grid_points, mu1_q, sigma1_hat)
    qda_g = qda_discriminant(grid_points, phi_q, mu0_q, mu1_q, sigma0_hat, sigma1_hat)

    x1_line = np.linspace(x1_min, x1_max, 100)

    plt.figure(figsize=(8.5, 7))
    plt.scatter(X[y == 0, 0], X[y == 0, 1], alpha=0.5, color="tab:blue", label="y=0")
    plt.scatter(X[y == 1, 0], X[y == 1, 1], alpha=0.5, color="tab:orange", label="y=1")

    plt.contour(x1_grid, x2_grid, pdf0, colors="tab:blue", linewidths=1, levels=6)
    plt.contour(x1_grid, x2_grid, pdf1, colors="tab:orange", linewidths=1, levels=6)

    plt.plot(x1_line, boundary_line(w_gda, b_gda, x1_line), color="black",
              linewidth=2, label="GDA boundary (shared covariance, linear)")
    plt.contour(x1_grid, x2_grid, qda_g, levels=[0], colors="red", linewidths=2,
                linestyles="solid")
    plt.plot([], [], color="red", linewidth=2, label="QDA boundary (separate covariances)")
    plt.plot(x1_line, boundary_line(w_log, b_log, x1_line), color="green",
              linewidth=2, linestyle="--", label="logistic regression boundary")

    plt.xlim(x1_min, x1_max)
    plt.ylim(x2_min, x2_max)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.title("GDA vs. QDA vs. logistic regression\n"
              "(true classes have different covariances)")
    plt.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.savefig("gda_qda_demo.png", dpi=150)
    print("Saved plot to gda_qda_demo.png")


if __name__ == "__main__":
    main()

## What if the data is not Gaussian?

In [ ]:
"""
Demo of what happens to GDA when its core assumption is violated: that
x | y is Gaussian. Here x is generated from a POISSON distribution
instead (independent Poisson counts per dimension, with a different
rate per class) - discrete, non-negative, right-skewed data that
looks nothing like a Gaussian, especially at the low counts used here.

GDA is still fit as usual, assuming x | y ~ N(mu, Sigma) - i.e. it is
deliberately misspecified. Logistic regression makes no distributional
assumption about x at all; it only models p(y|x) directly, which is
why it is largely unaffected by x's true distribution. Both are fit on
the same Poisson-generated data and compared:

  - visually: the fitted "Gaussian" density contours/marginals against
    the actual discrete, skewed histogram of the data - including
    Gaussian probability mass placed over negative values that
    Poisson data can never take
  - numerically: held-out accuracy and log-loss (probability
    calibration)

Note on the numeric comparison: for two-class problems where a good LINEAR 
separator exists (as it does here), classifiers based on a 
shared-covariance Gaussian assumption are often surprisingly robust to that 
assumption being violated - the resulting linear boundary can still be a 
reasonable one, even though the underlying density model is visibly wrong. 
The gap you'll see below in accuracy/log-loss is real but modest; the clearest, 
least ambiguous evidence that GDA's assumption has failed is the density
plot, not the classification score. This will however, affect the model's ability
to generate data.
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import warnings

N_PER_CLASS = 300
LAMBDA0 = np.array([1.0, 1.0])   # Poisson rate for class 0, each dimension
LAMBDA1 = np.array([3.0, 3.0])   # Poisson rate for class 1, each dimension
SEED = 0


def generate_data(n_per_class, lambda0, lambda1, rng):
    x0 = rng.poisson(lambda0, size=(n_per_class, 2)).astype(float)
    x1 = rng.poisson(lambda1, size=(n_per_class, 2)).astype(float)
    X = np.vstack([x0, x1])
    y = np.concatenate([np.zeros(n_per_class), np.ones(n_per_class)])
    return X, y


def fit_gda_shared_covariance(X, y):
    """Standard GDA MLE, assuming x | y ~ N(mu, Sigma) with Sigma
    shared across classes - fit here on data that does NOT actually
    come from a Gaussian, to see how well the assumption survives."""
    n = len(y)
    phi = np.mean(y == 1)
    mu0 = X[y == 0].mean(axis=0)
    mu1 = X[y == 1].mean(axis=0)
    centered = np.where((y == 1)[:, None], X - mu1, X - mu0)
    sigma = (centered.T @ centered) / n
    return phi, mu0, mu1, sigma


def gaussian_pdf(X_grid, mu, sigma):
    d = mu.shape[0]
    sigma_inv = np.linalg.inv(sigma)
    sigma_det = np.linalg.det(sigma)
    norm_const = 1.0 / (np.sqrt((2 * np.pi) ** d * sigma_det))
    diff = X_grid - mu
    exponent = -0.5 * np.einsum("...i,ij,...j->...", diff, sigma_inv, diff)
    return norm_const * np.exp(exponent)


def gda_linear_boundary_params(phi, mu0, mu1, sigma):
    sigma_inv = np.linalg.inv(sigma)
    w = sigma_inv @ (mu1 - mu0)
    b = (-0.5 * mu1 @ sigma_inv @ mu1
         + 0.5 * mu0 @ sigma_inv @ mu0
         + np.log(phi / (1 - phi)))
    return w, b


def gda_predict_proba(X, w, b):
    z = X @ w + b
    return 1.0 / (1.0 + np.exp(-z))


def boundary_line(w, b, x1_range):
    return -(w[0] * x1_range + b) / w[1]


def main():
    rng_train = np.random.default_rng(SEED)
    rng_test = np.random.default_rng(SEED + 1)

    X_train, y_train = generate_data(N_PER_CLASS, LAMBDA0, LAMBDA1, rng_train)
    X_test, y_test = generate_data(N_PER_CLASS, LAMBDA0, LAMBDA1, rng_test)

    # --- Fit GDA (misspecified: assumes Gaussian x | y, but x is Poisson) ---
    phi, mu0, mu1, sigma = fit_gda_shared_covariance(X_train, y_train)
    w_gda, b_gda = gda_linear_boundary_params(phi, mu0, mu1, sigma)
    print("GDA (assumes Gaussian x | y, but x is actually Poisson):")
    print(f"  mu0={mu0}, mu1={mu1}")
    print(f"  Sigma=\n{sigma}")

    p_gda_train = gda_predict_proba(X_train, w_gda, b_gda)
    p_gda_test = gda_predict_proba(X_test, w_gda, b_gda)
    gda_train_acc = np.mean((p_gda_train >= 0.5) == y_train)
    gda_test_acc = np.mean((p_gda_test >= 0.5) == y_test)
    gda_test_ll = log_loss(y_test, p_gda_test)
    print(f"  train accuracy = {gda_train_acc:.4f}, test accuracy = {gda_test_acc:.4f}, "
          f"test log-loss = {gda_test_ll:.4f}\n")

    # --- Fit logistic regression - no distributional assumption on x ---
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        logreg = LogisticRegression(C=np.inf)
        logreg.fit(X_train, y_train)
    w_log, b_log = logreg.coef_[0], logreg.intercept_[0]

    logreg_train_acc = logreg.score(X_train, y_train)
    logreg_test_acc = logreg.score(X_test, y_test)
    logreg_test_ll = log_loss(y_test, logreg.predict_proba(X_test)[:, 1])
    print("Logistic regression (no distributional assumption on x):")
    print(f"  train accuracy = {logreg_train_acc:.4f}, test accuracy = {logreg_test_acc:.4f}, "
          f"test log-loss = {logreg_test_ll:.4f}")
    print(f"\n(Gap is real but modest, as noted in the module docstring: a linear "
          f"boundary is fairly forgiving of this particular misspecification. "
          f"The density mismatch, shown in the plot, is the clearer failure.)")

    # --- Visualization ---
    rng_jitter = np.random.default_rng(SEED + 2)
    jitter = rng_jitter.uniform(-0.3, 0.3, X_train.shape)
    X_plot = X_train + jitter

    x1_min, x1_max = X_plot[:, 0].min() - 1, X_plot[:, 0].max() + 1
    x2_min, x2_max = X_plot[:, 1].min() - 1, X_plot[:, 1].max() + 1
    x1_grid, x2_grid = np.meshgrid(
        np.linspace(x1_min, x1_max, 300), np.linspace(x2_min, x2_max, 300)
    )
    grid_points = np.stack([x1_grid, x2_grid], axis=-1)
    pdf0 = gaussian_pdf(grid_points, mu0, sigma)
    pdf1 = gaussian_pdf(grid_points, mu1, sigma)
    x1_line = np.linspace(x1_min, x1_max, 100)

    fig, (ax_main, ax_hist) = plt.subplots(1, 2, figsize=(15, 6.5),
                                             gridspec_kw={"width_ratios": [1.3, 1]})

    # Left: scatter + GDA's Gaussian contours + both boundaries.
    ax_main.scatter(X_plot[y_train == 0, 0], X_plot[y_train == 0, 1], alpha=0.4,
                     color="tab:blue", s=15, label="y=0 (Poisson, jittered)")
    ax_main.scatter(X_plot[y_train == 1, 0], X_plot[y_train == 1, 1], alpha=0.4,
                     color="tab:orange", s=15, label="y=1 (Poisson, jittered)")
    ax_main.contour(x1_grid, x2_grid, pdf0, colors="tab:blue", linewidths=1, levels=6)
    ax_main.contour(x1_grid, x2_grid, pdf1, colors="tab:orange", linewidths=1, levels=6)
    ax_main.axvline(0, color="grey", linewidth=1, linestyle=":")
    ax_main.axhline(0, color="grey", linewidth=1, linestyle=":")
    ax_main.plot(x1_line, boundary_line(w_gda, b_gda, x1_line), color="black",
                 linewidth=2, label=f"GDA boundary (test acc={gda_test_acc:.3f})")
    ax_main.plot(x1_line, boundary_line(w_log, b_log, x1_line), color="green",
                 linewidth=2, linestyle="--",
                 label=f"logistic regression boundary (test acc={logreg_test_acc:.3f})")
    ax_main.set_xlim(x1_min, x1_max)
    ax_main.set_ylim(x2_min, x2_max)
    ax_main.set_xlabel("$x_1$")
    ax_main.set_ylabel("$x_2$")
    ax_main.set_title("GDA's fitted Gaussian contours vs. actual Poisson data\n"
                       "(dotted lines mark x=0: impossible for Poisson data, "
                       "routine for the Gaussian model)")
    ax_main.legend(loc="upper left", fontsize=8)

    # Right: marginal histogram of x1 for class 1, vs. GDA's fitted Gaussian
    # marginal - the clearest single picture of the assumption violation.
    x1_class1 = X_train[y_train == 1, 0]
    bins = np.arange(-0.5, x1_class1.max() + 1.5, 1)
    ax_hist.hist(x1_class1, bins=bins, density=True, alpha=0.6, color="tab:orange",
                 label="actual data: $x_1 \\mid y=1$ (Poisson)")
    x_line = np.linspace(x1_min, x1_max, 300)
    gaussian_marginal = (1 / np.sqrt(2 * np.pi * sigma[0, 0])
                         * np.exp(-0.5 * (x_line - mu1[0]) ** 2 / sigma[0, 0]))
    ax_hist.plot(x_line, gaussian_marginal, color="black", linewidth=2,
                 label="GDA's fitted Gaussian marginal")
    ax_hist.axvline(0, color="grey", linewidth=1, linestyle=":")
    ax_hist.set_xlabel("$x_1$")
    ax_hist.set_ylabel("density")
    ax_hist.set_title("Marginal mismatch: discrete/skewed data\nvs. GDA's assumed Gaussian shape")
    ax_hist.legend(fontsize=8)

    fig.tight_layout()
    fig.savefig("gda_poisson_demo.png", dpi=150)
    print("Saved plot to gda_poisson_demo.png")


if __name__ == "__main__":
    main()

## Strongly skewed priors - false postivies, false negatives

In [ ]:
"""
Demo of Gaussian Discriminant Analysis (GDA) vs. logistic regression
on two-dimensional data with two classes y in {0, 1}, under a heavily
SKEWED class prior: P(y=1) = P_Y1 <<0.5, rather
than the balanced 50/50 split used in the basic GDA demo.

Model (standard GDA, shared covariance across classes), same as the
balanced-prior demo:
    y ~ Bernoulli(phi)
    x | y=0 ~ N(mu0, Sigma)
    x | y=1 ~ N(mu1, Sigma)

The data-generating process drawns from each class: 
n1 = round(N_TOTAL * P_Y1), n0 = N_TOTAL - n1.
GDA's own phi is then estimated from this, and the class prior enters 
its decision boundary directly through the log(phi/(1-phi))
term - a heavily skewed prior pulls that term strongly negative,
shifting the boundary to make positive predictions "harder to reach"
and, in the process, reducing false positives at the cost of missing
more true positives (false negatives). Logistic regression's fitted
intercept absorbs the same prior imbalance, though via a different
estimation route (direct discriminative MLE rather than the explicit
Bayes'-rule prior term).

A separate held-out test set, generated with the same prior, is
used to report false positives (FP) and false negatives (FN) for both
classifiers - the metrics that matter most for a rare-class problem,
since overall accuracy is a poor/misleading measure here (predicting
"always y=0" already gets ~99% accuracy).
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
import warnings

N_TOTAL = 5000       # total points, split according to P_Y1
P_Y1 = 0.03           # P(y=1): a heavily skewed, rare-positive prior
MU0 = np.array([-1.0, -1.0])
MU1 = np.array([1.8, 1.8])
SIGMA = np.array([[2.0, 0.8],
                   [0.8, 1.5]])   # shared covariance for both classes
SEED = 0


def generate_data(n_total, p_y1, mu0=MU0, mu1=MU1, sigma=SIGMA, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    n1 = round(n_total * p_y1)
    n0 = n_total - n1
    x0 = rng.multivariate_normal(mu0, sigma, n0)
    x1 = rng.multivariate_normal(mu1, sigma, n1)
    X = np.vstack([x0, x1])
    y = np.concatenate([np.zeros(n0), np.ones(n1)])
    return X, y


def fit_gda(X, y):
    n = len(y)
    phi = np.mean(y == 1)
    mu0 = X[y == 0].mean(axis=0)
    mu1 = X[y == 1].mean(axis=0)

    centered = np.where((y == 1)[:, None], X - mu1, X - mu0)
    Sigma = (centered.T @ centered) / n

    return phi, mu0, mu1, Sigma


def gaussian_pdf(X_grid, mu, sigma):
    d = mu.shape[0]
    sigma_inv = np.linalg.inv(sigma)
    sigma_det = np.linalg.det(sigma)
    norm_const = 1.0 / (np.sqrt((2 * np.pi) ** d * sigma_det))
    diff = X_grid - mu
    exponent = -0.5 * np.einsum("...i,ij,...j->...", diff, sigma_inv, diff)
    return norm_const * np.exp(exponent)


def gda_decision_boundary_params(phi, mu0, mu1, sigma):
    sigma_inv = np.linalg.inv(sigma)
    w = sigma_inv @ (mu1 - mu0)
    b = (-0.5 * mu1 @ sigma_inv @ mu1
         + 0.5 * mu0 @ sigma_inv @ mu0
         + np.log(phi / (1 - phi)))
    return w, b


def boundary_line(w, b, x1_range):
    return -(w[0] * x1_range + b) / w[1]


def confusion_counts(y_true, y_pred):
    """Return (TP, FP, FN, TN)."""
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    return tp, fp, fn, tn


def main():
    rng_train = np.random.default_rng(SEED)
    rng_test = np.random.default_rng(SEED + 1)

    X, y = generate_data(N_TOTAL, P_Y1, rng=rng_train)
    X_test, y_test = generate_data(N_TOTAL, P_Y1, rng=rng_test)
    print(f"Training set: {int((y == 0).sum())} of class 0, {int((y == 1).sum())} of class 1 "
          f"(P(y=1) target = {P_Y1})")

    phi, mu0, mu1, Sigma = fit_gda(X, y)
    print(f"\nGDA estimates:")
    print(f"  phi = {phi:.4f}  (target P(y=1) = {P_Y1})")
    print(f"  mu0 = {mu0}")
    print(f"  mu1 = {mu1}")
    print(f"  Sigma =\n{Sigma}")

    w_gda, b_gda = gda_decision_boundary_params(phi, mu0, mu1, Sigma)
    print(f"GDA decision boundary: w={w_gda}, b={b_gda:.4f}")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        logreg = LogisticRegression(C=np.inf)
        logreg.fit(X, y)
    w_log = logreg.coef_[0]
    b_log = logreg.intercept_[0]
    print(f"Logistic regression boundary: w={w_log}, b={b_log:.4f}")

    # --- Evaluate both classifiers on the held-out test set ---
    y_pred_gda = (X_test @ w_gda + b_gda >= 0).astype(int)
    y_pred_log = logreg.predict(X_test)

    tp_g, fp_g, fn_g, tn_g = confusion_counts(y_test, y_pred_gda)
    tp_l, fp_l, fn_l, tn_l = confusion_counts(y_test, y_pred_log)

    print(f"\nTest set: {int((y_test == 0).sum())} of class 0, "
          f"{int((y_test == 1).sum())} of class 1")
    print(f"GDA:                 TP={tp_g}  FP={fp_g}  FN={fn_g}  TN={tn_g}  "
          f"(accuracy={(tp_g+tn_g)/len(y_test):.4f})")
    print(f"Logistic regression: TP={tp_l}  FP={fp_l}  FN={fn_l}  TN={tn_l}  "
          f"(accuracy={(tp_l+tn_l)/len(y_test):.4f})")

    # --- Plot ---
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    x1_grid, x2_grid = np.meshgrid(
        np.linspace(x1_min, x1_max, 200), np.linspace(x2_min, x2_max, 200)
    )
    grid_points = np.stack([x1_grid, x2_grid], axis=-1)
    pdf0 = gaussian_pdf(grid_points, mu0, Sigma)
    pdf1 = gaussian_pdf(grid_points, mu1, Sigma)
    x1_line = np.linspace(x1_min, x1_max, 100)

    plt.figure(figsize=(8.5, 7))
    plt.scatter(X[y == 0, 0], X[y == 0, 1], alpha=0.15, color="tab:blue",
                s=10, label=f"y=0 (n={int((y==0).sum())})")
    plt.scatter(X[y == 1, 0], X[y == 1, 1], alpha=0.7, color="tab:orange",
                s=15, label=f"y=1 (n={int((y==1).sum())})")

    plt.contour(x1_grid, x2_grid, pdf0, colors="tab:blue", linewidths=1, levels=6)
    plt.contour(x1_grid, x2_grid, pdf1, colors="tab:orange", linewidths=1, levels=6)

    plt.plot(x1_line, boundary_line(w_gda, b_gda, x1_line), color="black",
              linewidth=2, label=f"GDA boundary (FP={fp_g}, FN={fn_g})")
    plt.plot(x1_line, boundary_line(w_log, b_log, x1_line), color="green",
              linewidth=2, linestyle="--",
              label=f"logistic regression boundary (FP={fp_l}, FN={fn_l})")

    plt.xlim(x1_min, x1_max)
    plt.ylim(x2_min, x2_max)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.title(f"GDA vs. logistic regression under a skewed prior "
              f"(P(y=1)={P_Y1})\ntest-set false positives / false negatives annotated")
    plt.legend(loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.savefig("gda_skewed_prior_demo.png", dpi=150)
    print("\nSaved plot to gda_skewed_prior_demo.png")


if __name__ == "__main__":
    main()